# MixConfig MNIST demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Q9gJYx/MixConfig/blob/main/notebooks/demo_mnist.ipynb) [![ICML 2026](https://img.shields.io/badge/ICML-2026-1d4ed8.svg)](https://icml.cc/virtual/2026/poster/63010) [![arXiv](https://img.shields.io/badge/arXiv-2510.19248-b31b1b.svg)](https://arxiv.org/abs/2510.19248) [![OpenReview](https://img.shields.io/badge/OpenReview-aw6alulxr8-8c1b13.svg)](https://openreview.net/forum?id=aw6alulxr8)

Smoke test for the pre-extracted MixConfig artifacts on MNIST-70k.

**What this notebook does:**
1. Loads `data/mnist_configs.npz` (configurations + energy statistics from BlueRed front + HOG features).
2. Validates the artifact via `src.mixconfig.validate_configurations`.
3. Inspects the BlueRed-front structure (9 stable configurations spanning k=1 coarse to k=70000 singletons; the highest-resolution non-trivial configuration is k=9 with persistence λ ≈ 0.957, which approximates but does not exactly match the 10 MNIST digit classes - the bluered 10-cluster match is the n=5000 subset, not the 70k variant we ship here).
4. Builds the Energy-Aware Selector and runs a forward pass on a 1000-sample subset, dropping the singleton-endpoint configuration.

Runtime: under 60 seconds on Colab CPU. No GPU needed for this smoke test.

For full reproduction commands across tabular / vision / molecular / text benchmarks, see [README.md](https://github.com/Q9gJYx/MixConfig/blob/main/README.md#reproducing-paper-results).

In [ ]:
# Colab setup: clone the repo and put it on the Python path.
# Local execution: skip this cell (the parent dir is already on the path).
import os, sys
if not os.path.exists("MixConfig") and not os.path.exists("../src/mixconfig"):
    !git clone -q https://github.com/Q9gJYx/MixConfig.git
REPO = "MixConfig" if os.path.exists("MixConfig") else ".."
sys.path.insert(0, REPO)
print(f"Using repo root: {REPO}")

In [ ]:
import numpy as np

bundle = np.load(f"{REPO}/data/mnist_configs.npz")
print("Keys in the bundle:")
for k in bundle.files:
    print(f"  {k}: shape={bundle[k].shape} dtype={bundle[k].dtype}")

In [ ]:
from src.mixconfig import validate_configurations

configs = bundle["configs"]
energy_stats = bundle["energy_stats"]
validate_configurations(configs, energy_stats)
print("validate_configurations: OK")

n_clusters = [int(len(np.unique(configs[:, j]))) for j in range(configs.shape[1])]
print(f"\nBlueRed front: {configs.shape[1]} configurations on n={configs.shape[0]:,} MNIST samples")
print(f"Cluster counts per config: {n_clusters}")
print("\nEnergy statistics (rows = configs, cols = [H, h_a, h_r, delta_gamma]):")
with np.printoptions(precision=3, suppress=True):
    print(energy_stats)

In [ ]:
# Forward pass with the Energy-Aware Selector.
# - Drop the singleton-endpoint config (k = 70000) before feeding the selector:
#   the bounded ClusterAssignmentEmbedder has one embedding table per config, sized
#   by max_clusters. A 70000-row table per config is wasteful and uninformative.
# - Features here are raw MNIST pixels (784-dim) on a 1000-sample subset, for speed.
#   In a downstream task you'd use whatever embedding makes sense (CLIP, BERT, GIN, etc.).

import torch
from sklearn.datasets import fetch_openml
from src.mixconfig import EnergyAwareSelector

keep = configs.shape[1] - 1                 # drop the trivial singleton endpoint
configs_kept = configs[:, :keep]
energy_kept = energy_stats[:keep]
max_k = int(configs_kept.max()) + 1
print(f"Keeping {keep}/{configs.shape[1]} configs; max cluster id = {max_k - 1}")

print("Fetching MNIST features (cached on second run)...")
mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="liac-arff")
X = mnist.data[:1000].astype(np.float32) / 255.0
configs_sub = configs_kept[:1000].astype(np.int64)

selector = EnergyAwareSelector(
    input_dim=X.shape[1],
    n_configs=keep,
    max_clusters=max_k,
    context_dim=64,
    cluster_embed_dim=32,
)
selector.eval()

with torch.no_grad():
    z = selector.get_mixed_representation(
        torch.tensor(X),
        torch.tensor(configs_sub, dtype=torch.long),
        torch.tensor(energy_kept, dtype=torch.float32),
    )
print(f"\nMixed representation: shape={tuple(z.shape)} dtype={z.dtype}")
print(f"Finite: {torch.isfinite(z).all().item()}  mean: {z.mean().item():.4f}  std: {z.std().item():.4f}")
print("Smoke test passed.")

## Next steps

- Plug `z` into any downstream classifier (linear probe, XGBoost, MLP) and compare against single-configuration baselines.
- Swap the raw-pixel features above for your preferred embedding; the selector recomputes energy statistics internally when you pass `energy_stats=None`.
- For the full set of benchmark scripts and ablations, see [`experiments/`](https://github.com/Q9gJYx/MixConfig/tree/main/experiments) in the repo.
- For the substitution interface (use your own multi-resolution clustering routine), see [`docs/configurations.md`](https://github.com/Q9gJYx/MixConfig/blob/main/docs/configurations.md).